# Cerebras

In [1]:
import os
from cerebras.cloud.sdk import Cerebras

client_cerebras = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

In [2]:
from pydantic import BaseModel
import json
from typing import Literal
from pydantic import BaseModel, Field

class SupervisorResponse(BaseModel):
    valid: Literal[True, False] = Field(description="Valid from the supervisor, if not is need more interactions return True, else return False")
    feedback: str = Field(description="Feedback from the supervisor")

# Convert the Pydantic model to a JSON schema
movie_schema = SupervisorResponse.model_json_schema()

# Print the JSON schema to verify it
print(json.dumps(movie_schema, indent=2))

{
  "properties": {
    "valid": {
      "description": "Valid from the supervisor, if not is need more interactions return True, else return False",
      "enum": [
        true,
        false
      ],
      "title": "Valid",
      "type": "boolean"
    },
    "feedback": {
      "description": "Feedback from the supervisor",
      "title": "Feedback",
      "type": "string"
    }
  },
  "required": [
    "valid",
    "feedback"
  ],
  "title": "SupervisorResponse",
  "type": "object"
}


In [3]:
from typing import Optional
from pydantic import BaseModel
from cerebras.cloud.sdk import AsyncCerebras
import os
from dotenv import load_dotenv

load_dotenv()

import logging

# Configurar logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


client_cerebras = AsyncCerebras(
)

class RouterCerebras:
    def __init__(self,  messages: str, model_llm: str, strutured_output: Optional[BaseModel] = None):
        self.messages = messages
        self.model_llm = model_llm
        self.strutured_output = strutured_output
        
    async def get_response_cerebras_structured_async(self) -> Optional[BaseModel]:
        if self.strutured_output is None:
            raise ValueError(
                    "structured_output precisa estar definido para usar essa função."
                )
        try:
            response = await client_cerebras.chat.completions.create(
                    model=self.model_llm,
                    messages=[{"role": "user", "content": self.messages}],
                    response_format={
                        "type": "json_schema",
                        "json_schema": {
                            "name": "structured_response",
                            "schema": self.strutured_output.model_json_schema(),
                        }
                    }
                )
            return response.choices[0].message.content
        except Exception as e:
            logger.error(f"Erro no llm_pydanticai: {e}")
            raise
        
    async def get_response_cerebras_async(self) -> str:
        try:
            response = await client_cerebras.chat.completions.create(
                    model=self.model_llm,
                    messages=[{"role": "user", "content": self.messages}],
                )
            return response.choices[0].message.content
        except Exception as e:
            logger.error(f"Erro no llm_pydanticai: {e}")
            raise
        

INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"


In [4]:
router = RouterCerebras(
    messages="Explique a importância da inferência rápida.",
    model_llm="qwen-3-235b-a22b-instruct-2507",
    strutured_output = SupervisorResponse
)

In [5]:
await router.get_response_cerebras_structured_async()

INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"


'{"valid": false, "feedback": "A solicitação é para explicar a importância da inferência rápida, mas a resposta anterior não foi fornecida. Por favor, forneça a explicação para que eu possa avaliar a validade e dar feedback adequado."}'

In [6]:
from pydantic import BaseModel
import json
from typing import Literal
from pydantic import BaseModel, Field

class SupervisorResponse(BaseModel):
    valid: Literal[True, False] = Field(description="Valid from the supervisor, if not is need more interactions return True, else return False")
    feedback: str = Field(description="Feedback from the supervisor")

# Convert the Pydantic model to a JSON schema
movie_schema = SupervisorResponse.model_json_schema()

# Print the JSON schema to verify it
print(json.dumps(movie_schema, indent=2))

{
  "properties": {
    "valid": {
      "description": "Valid from the supervisor, if not is need more interactions return True, else return False",
      "enum": [
        true,
        false
      ],
      "title": "Valid",
      "type": "boolean"
    },
    "feedback": {
      "description": "Feedback from the supervisor",
      "title": "Feedback",
      "type": "string"
    }
  },
  "required": [
    "valid",
    "feedback"
  ],
  "title": "SupervisorResponse",
  "type": "object"
}


In [7]:
import asyncio
from cerebras.cloud.sdk import Cerebras, AsyncCerebras


class GetModelCerebrasStructuredResponse:
    def client(self) -> Cerebras:
        return Cerebras(
            api_key=os.environ.get("CEREBRAS_API_KEY")
        )
    
    def client_async(self) -> AsyncCerebras:
        return AsyncCerebras(
            api_key=os.environ.get("CEREBRAS_API_KEY")
        )
    
    def models_cerebras(self) -> list[str]:
        client_cerebras = self.client()
        models = client_cerebras.models.list()
        return [model.id for model in models.data if not "thinking" in model.id if not "coder" in model.id]
    
    def get_models_cerebras_structured(self) -> list[str]:
        models_vailable_cerebras = self.models_cerebras()
        client_cerebras = self.client()
        model_cerebras_strured = []
        for model in models_vailable_cerebras:
            try:
                completion = client_cerebras.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": "You are a helpful assistant that generates movie recommendations."},
                        {"role": "user", "content": "Suggest a sci-fi movie from the 1990s"}
                    ],
                    response_format={
                        "type": "json_schema", 
                        "json_schema": {
                            "name": "structured_response",
                            "strict": True,
                            "schema": SupervisorResponse.model_json_schema()
                        }
                    }
                )
                model_cerebras_strured.append(model)
            except Exception as _:
                pass
        return model_cerebras_strured
    
    async def get_models_cerebras_structured_async(self) -> list[str]:
        
        models_vailable_cerebras = self.models_cerebras()
        client_cerebras = self.client_async()
        async def check_model(model: str) -> str | None:
            try:
                completion = await client_cerebras.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": "You are a helpful assistant that generates movie recommendations."},
                        {"role": "user", "content": "Suggest a sci-fi movie from the 1990s"}
                    ],
                    response_format={
                        "type": "json_schema", 
                        "json_schema": {
                            "name": "structured_response",
                            "strict": True,
                            "schema": movie_schema
                        }
                    }
                )
                return model
            except Exception as _:
                return None
        
        # Executa todas as verificações em paralelo
        results = await asyncio.gather(*[check_model(model) for model in models_vailable_cerebras])
        
        model_cerebras_structured = [model for model in results if model is not None]
        
        return model_cerebras_structured    
        
        

In [ ]:
cerebras_models = GetModelCerebrasStructuredResponse()

cerebras_models.get_models_cerebras_structured()

In [9]:
await cerebras_models.get_models_cerebras_structured_async()

INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/models "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"


['llama-4-maverick-17b-128e-instruct',
 'qwen-3-235b-a22b-instruct-2507',
 'qwen-3-32b',
 'gpt-oss-120b',
 'llama3.1-8b',
 'llama-4-scout-17b-16e-instruct',
 'llama-3.3-70b']

In [30]:
cerebras_models.models_cerebras()

INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/models "HTTP/1.1 200 OK"


['gpt-oss-120b',
 'qwen-3-235b-a22b-instruct-2507',
 'llama-4-maverick-17b-128e-instruct',
 'llama-4-scout-17b-16e-instruct',
 'qwen-3-32b',
 'llama3.1-8b',
 'llama-3.3-70b']

## Groq

In [ ]:
import asyncio
from groq import Groq, AsyncGroq
from typing import Literal
from pydantic import BaseModel, Field

class SupervisorResponse(BaseModel):
    valid: Literal[True, False] = Field(description="Valid from the supervisor, if not is need more interactions return True, else return False")
    feedback: str = Field(description="Feedback from the supervisor")


class GetModelGroqStructuredResponse:
    def client(self) -> Groq:
        return Groq()
    
    def client_async(self) -> AsyncGroq:
        return AsyncGroq()
    
    def models_groq(self) -> list[str]:
        client_groq = self.client()
        models = client_groq.models.list()
        return [model.id for model in models.data if not "thinking" in model.id if not "coder" in model.id]
    
    def get_models_groq_structured(self) -> list[str]:
        models_vailable_groq = self.models_groq()
        client_groq = self.client()
        model_groq_strured = []
        for model in models_vailable_groq:
            try:
                _ = client_groq.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": "You are a helpful assistant that generates movie recommendations."},
                        {"role": "user", "content": "Suggest a sci-fi movie from the 1990s"}
                    ],
                    response_format={
                        "type": "json_schema",
                        "json_schema": {
                            "name": "structured_response",
                            "schema": SupervisorResponse.model_json_schema(),
                        }
                    }
                )
                model_groq_strured.append(model)
            except Exception as _:
                pass
        return model_groq_strured
    
    async def get_models_groq_structured_async(self) -> list[str]:
        models_available_groq = self.models_groq()  # Se models_groq também for async
        client_groq = self.client_async()
        async def check_model(model: str) -> str | None:
            try:
                completion = await client_groq.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": "You are a helpful assistant that generates movie recommendations."},
                        {"role": "user", "content": "Suggest a sci-fi movie from the 1990s"}
                    ],
                    response_format={
                        "type": "json_schema",
                        "json_schema": {
                            "name": "structured_response",
                            "schema": movie_schema,
                        }
                    }
                )
                return model
            except Exception as _:
                return None
        
        # Executa todas as verificações em paralelo
        results = await asyncio.gather(*[check_model(model) for model in models_available_groq])
        
        model_groq_structured = [model for model in results if model is not None]
        
        return model_groq_structured


get_models_structured = GetModelGroqStructuredResponse()
models_vailable_groq = await get_models_structured.get_models_groq_structured_async()



In [29]:
get_models_structured.models_groq()

INFO:httpx:HTTP Request: GET https://api.groq.com/openai/v1/models "HTTP/1.1 200 OK"


['llama-3.1-8b-instant',
 'meta-llama/llama-prompt-guard-2-22m',
 'whisper-large-v3-turbo',
 'llama-3.3-70b-versatile',
 'qwen/qwen3-32b',
 'meta-llama/llama-4-maverick-17b-128e-instruct',
 'whisper-large-v3',
 'meta-llama/llama-guard-4-12b',
 'groq/compound-mini',
 'openai/gpt-oss-20b',
 'moonshotai/kimi-k2-instruct-0905',
 'meta-llama/llama-prompt-guard-2-86m',
 'playai-tts',
 'playai-tts-arabic',
 'deepseek-r1-distill-llama-70b',
 'moonshotai/kimi-k2-instruct',
 'groq/compound',
 'allam-2-7b',
 'openai/gpt-oss-120b',
 'meta-llama/llama-4-scout-17b-16e-instruct']

## Nvidia

In [12]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA()

In [63]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from typing import Any

class GetModelsNvidia:
    def client(self) -> ChatNVIDIA:
        return ChatNVIDIA()
    
    def models_nvidia(self) -> list[Any]:
        client_nvidia = self.client()
        models = client_nvidia.available_models
        return models
    
    def name_all_models_nvidia(self) -> list[Any]:
        models_vailable_nvidia = self.models_nvidia()
        models = [model.id for model in models_vailable_nvidia]
        return models
    
    def get_models_nvidia_chat(self) -> list[Any]:
        models_vailable_nvidia = self.models_nvidia()
        model_nvidia_chat = [model.id for model in models_vailable_nvidia if model.model_type == "chat"]
        model_nvidia_chat_sem_code = [model for model in model_nvidia_chat if "code" not in model]
        return model_nvidia_chat_sem_code
    
    def get_models_nvidia_structured(self) -> list[Any]:
        models_vailable_nvidia = self.models_nvidia()
        model_nvidia_strured = [model.id for model in models_vailable_nvidia if model.supports_structured_output]
        return model_nvidia_strured
    
    def get_models_nvidia_tools(self) -> list[Any]:
        models_vailable_nvidia = self.models_nvidia()
        model_nvidia_tools = [model.id for model in models_vailable_nvidia if model.supports_tools]
        return model_nvidia_tools
    def get_models_nvidia_coder(self) -> list[Any]:
        models_vailable_nvidia = self.models_nvidia()
        model_nvidia_coder = [model.id for model in models_vailable_nvidia if "code" in model.id]
        return model_nvidia_coder

In [64]:
nivida_models = GetModelsNvidia()

In [65]:
all_nivida_models = GetModelsNvidia().get_models_nvidia_tools()

In [68]:
nivida_models.get_models_nvidia_chat()

['thudm/chatglm3-6b',
 'nvidia/nemotron-4-mini-hindi-4b-instruct',
 'baichuan-inc/baichuan2-13b-chat',
 'nvidia/llama-3.1-nemotron-nano-8b-v1',
 'microsoft/phi-3.5-moe-instruct',
 'tokyotech-llm/llama-3-swallow-70b-instruct-v0.1',
 'moonshotai/kimi-k2-instruct',
 'meta/llama-guard-4-12b',
 'nvidia/riva-translate-4b-instruct',
 'deepseek-ai/deepseek-r1',
 'nvidia/llama-3.3-nemotron-super-49b-v1.5',
 'ibm/granite-3.0-3b-a800m-instruct',
 'snowflake/arctic',
 'microsoft/phi-3-mini-128k-instruct',
 'gotocompany/gemma-2-9b-cpt-sahabatai-instruct',
 'nv-mistralai/mistral-nemo-12b-instruct',
 'microsoft/phi-4-mini-instruct',
 'qwen/qwen2.5-7b-instruct',
 'microsoft/phi-3-medium-4k-instruct',
 'google/gemma-3-4b-it',
 'deepseek-ai/deepseek-r1-distill-qwen-32b',
 'meta/llama-3.3-70b-instruct',
 'nvidia/llama-3.1-nemotron-ultra-253b-v1',
 'deepseek-ai/deepseek-r1-distill-llama-8b',
 'google/gemma-7b',
 'nvidia/nemotron-4-340b-instruct',
 'google/gemma-2b',
 'ai21labs/jamba-1.5-large-instruct',
 

## All Models

In [ ]:
from typing import Dict, List
class GetModels:
    def __init__(self) -> None:
        self.nvidia = GetModelsNvidia()
        self.groq = GetModelGroqStructuredResponse()
        self.cerebras = GetModelCerebrasStructuredResponse()
        
    def nvidia_structured(self) -> List[Dict[str, str]]:
        nvidia_structured = self.nvidia.get_models_nvidia_structured()
        list_nvidia_structured = [{"model": model, "provider": "nvidia"} for model in nvidia_structured]
        return  list_nvidia_structured
    
    def nvidia_tools(self) -> List[Dict[str, str]]:
        nvidia_tools = self.nvidia.get_models_nvidia_tools()
        list_nvidia_tools = [{"model": model, "provider": "nvidia"} for model in nvidia_tools]
        return list_nvidia_tools
    
    def nvidia_coder(self) -> List[Dict[str, str]]:
        nvidia_coder = self.nvidia.get_models_nvidia_coder()
        list_nvidia_coder = [{"model": model, "provider": "nvidia"} for model in nvidia_coder]
        return list_nvidia_coder
    
    def nvidia_chat(self) -> List[Dict[str, str]]:
        nvidia_chat = self.nvidia.get_models_nvidia_chat()
        list_nvidia_chat = [{"model": model, "provider": "nvidia"} for model in nvidia_chat]
        return list_nvidia_chat
    
    def groq_structured(self) -> List[Dict[str, str]]:
        groq_structured = self.groq.get_models_groq_structured()
        list_groq_structured = [{"model": model, "provider": "groq"} for model in groq_structured]
        return list_groq_structured
        
    def cerebras_structured(self) -> List[Dict[str, str]]:
        cerebras_structured = self.cerebras.get_models_cerebras_structured()
        list_cerebras_structured = [{"model": model, "provider": "cerebras"} for model in cerebras_structured]
        return list_cerebras_structured
    
    def all_model_nvidia(self) -> List[Dict[str, str]]:
        all_model_nvidia = self.nvidia.name_all_models_nvidia()
        list_all_model_nvidia = [{"model": model, "provider": "nvidia"} for model in all_model_nvidia]
        return list_all_model_nvidia
    
    def all_model_groq(self) -> List[Dict[str,str]]:
        all_model_groq = self.groq.models_groq()
        list_all_model_groq = [{"model": model, "provider": "groq"} for model in all_model_groq]
        return list_all_model_groq
    
    def all_models_cerebras(self) -> List[Dict[str,str]]:
        all_model_cerebras = self.cerebras.models_cerebras()
        list_all_model_groq = [{"model": model, "provider": "cerebras"} for model in all_model_cerebras]
        return list_all_model_groq
    
    

In [ ]:
models = GetModels()
all_models = models.all_model_nvidia()
all_models

In [ ]:
import asyncio
from typing import Dict, List


class GetModels:
    def __init__(self) -> None:
        self.nvidia = GetModelsNvidia()
        self.groq = GetModelGroqStructuredResponse()
        self.cerebras = GetModelCerebrasStructuredResponse()
        
    async def nvidia_structured(self) -> List[Dict[str, str]]:
        nvidia_structured = await asyncio.to_thread(self.nvidia.get_models_nvidia_structured)
        list_nvidia_structured = [{"model": model, "provider": "nvidia"} for model in nvidia_structured]
        return list_nvidia_structured
    
    async def nvidia_tools(self) -> List[Dict[str, str]]:
        nvidia_tools = await asyncio.to_thread(self.nvidia.get_models_nvidia_tools)
        list_nvidia_tools = [{"model": model, "provider": "nvidia"} for model in nvidia_tools]
        return list_nvidia_tools
    
    async def nvidia_coder(self) -> List[Dict[str, str]]:
        nvidia_coder = await asyncio.to_thread(self.nvidia.get_models_nvidia_coder)
        list_nvidia_coder = [{"model": model, "provider": "nvidia"} for model in nvidia_coder]
        return list_nvidia_coder
    
    async def nvidia_chat(self) -> List[Dict[str, str]]:
        nvidia_chat = await asyncio.to_thread(self.nvidia.get_models_nvidia_chat)
        list_nvidia_chat = [{"model": model, "provider": "nvidia"} for model in nvidia_chat]
        return list_nvidia_chat
    
    async def groq_structured(self) -> List[Dict[str, str]]:
        groq_structured = await asyncio.to_thread(self.groq.get_models_groq_structured)
        list_groq_structured = [{"model": model, "provider": "groq"} for model in groq_structured]
        return list_groq_structured
        
    async def cerebras_structured(self) -> List[Dict[str, str]]:
        cerebras_structured = await asyncio.to_thread(self.cerebras.get_models_cerebras_structured)
        list_cerebras_structured = [{"model": model, "provider": "cerebras"} for model in cerebras_structured]
        return list_cerebras_structured
    
    async def all_model_nvidia(self) -> List[Dict[str, str]]:
        all_model_nvidia = await asyncio.to_thread(self.nvidia.name_all_models_nvidia)
        list_all_model_nvidia = [{"model": model, "provider": "nvidia"} for model in all_model_nvidia]
        return list_all_model_nvidia
    
    async def all_model_groq(self) -> List[Dict[str, str]]:
        all_model_groq = await asyncio.to_thread(self.groq.models_groq)
        list_all_model_groq = [{"model": model, "provider": "groq"} for model in all_model_groq]
        return list_all_model_groq
    
    async def all_models_cerebras(self) -> List[Dict[str, str]]:
        all_model_cerebras = await asyncio.to_thread(self.cerebras.models_cerebras)
        list_all_model_cerebras = [{"model": model, "provider": "cerebras"} for model in all_model_cerebras]
        return list_all_model_cerebras
    
    async def get_all_models_async(self) -> Dict[str, List[Dict[str, str]] | BaseException]:
        """Busca todos os modelos de forma assíncrona"""
        results = await asyncio.gather(
            self.nvidia_structured(),
            self.nvidia_tools(),
            self.nvidia_coder(),
            self.nvidia_chat(),
            self.groq_structured(),
            self.cerebras_structured(),
            self.all_model_nvidia(),
            self.all_model_groq(),
            self.all_models_cerebras(),
            return_exceptions=True
        )
        
        return {
            "nvidia_structured": results[0] if not isinstance(results[0], Exception) else results[0],
            "nvidia_tools": results[1] if not isinstance(results[1], Exception) else results[1],
            "nvidia_coder": results[2] if not isinstance(results[2], Exception) else results[2],
            "nvidia_chat": results[3] if not isinstance(results[3], Exception) else results[3],
            "groq_structured": results[4] if not isinstance(results[4], Exception) else results[4],
            "cerebras_structured": results[5] if not isinstance(results[5], Exception) else results[5],
            "all_nvidia": results[6] if not isinstance(results[6], Exception) else results[6],
            "all_groq": results[7] if not isinstance(results[7], Exception) else results[7],
            "all_cerebras": results[8] if not isinstance(results[8], Exception) else results[8]
        }
        
    async def salvar_modelos(self, path:str)-> None:
        all_models = await self.get_all_models_async()
        with open(f"{path}/all_models.json", "w") as f:
            json.dump(all_models, f, indent=4)
        print("Dados salvos com sucess!")
        
    



In [89]:
all_models = GetModels()

In [90]:
await all_models.salvar_modelos("../data")

INFO:httpx:HTTP Request: GET https://api.groq.com/openai/v1/models "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.groq.com/openai/v1/models "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/models "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/models "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"

Dados salvos com sucess!


In [84]:
with open("../data/all_models.json", "w") as f:
    json.dump(all_models, f, indent=4)
